# OtterMap Challenge: Semantic Segmentation Pipeline

## 1. Project Workflow
This pipeline is optimized for efficiency on Google Colab by leveraging high-speed local disk I/O and pre-trained deep learning architectures.

### Data Ingestion & Acceleration
To overcome Google Drive latency, we migrate our dataset to the local Colab disk. We use `shutil.copytree` to move train and validation sets, drastically reducing I/O overhead during training epochs.

### Pre-processing
* **Geospatial Tools:** `GeoPandas` and `Rasterio` manage coordinate systems and spatial alignment.
* **Rasterization:** Vector GeoJSON labels are converted into binary pixel masks.
* **Patch Tiling:** High-resolution imagery is sliced into $256 \times 512$ tiles to optimize GPU memory utilization.

### Data Quality & "Quarantine"
* **Label Noise Mitigation:** We utilize a verification script to identify and quarantine mislabeled samples (e.g., images with empty masks).
* **Quality Assurance:** Only high-fidelity, validated image-mask pairs are used for model training.

---

## 2. Model Training & Augmentation

### Augmentation Strategy
To ensure model generalization, we employ dynamic augmentation (random rotations, horizontal flips, and cropping) during the loading process, effectively increasing dataset variety.

### Architecture & Fine-Tuning
We utilize `segmentation-models-pytorch` to perform transfer learning. By leveraging pre-trained ImageNet encoders (e.g., ResNet34), we achieve robust feature extraction even with limited aerial training data.

---

## 3. Evaluation & Persistence

### Monitoring
The training loop monitors validation loss, automatically saving `best_model.pth` to Google Drive to ensure persistence.

### Metrics
We evaluate performance using **mIoU (mean Intersection over Union)**.

### Performance Summary
* **Target Metric:** mIoU
* **Validation Strategy:** Unseen test set inference
* **Robustness:** Documented performance in shadow-heavy and mixed-vegetation environments.

###Mount To Google drive

In [2]:
from google.colab import drive
import os

drive.flush_and_unmount()
drive.mount('/content/drive')

project processes raw geospatial satellite imagery (TIF) and vector labels (GeoJSON) into high-quality training patches


*   CRS Alignment: We utilize geopandas and rasterio to align imagery and vector labels into a unified coordinate system
*   Rasterization: Vector polygons are converted into binary pixel masks using rasterio.features.rasterize.



In [2]:
import os
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import numpy as np
import cv2

PROJECT_ROOT = "/content/drive/MyDrive/ottermap_challenge"
RAW_DIR = os.path.join(PROJECT_ROOT, "data/raw")
PATCH_IMG_DIR = os.path.join(PROJECT_ROOT, "data/patches/images")
PATCH_MASK_DIR = os.path.join(PROJECT_ROOT, "data/patches/masks")

os.makedirs(PATCH_IMG_DIR, exist_ok=True)
os.makedirs(PATCH_MASK_DIR, exist_ok=True)

parcels = ["1", "2", "3"]
patch_h, patch_w = 256, 512
total_patches_saved = 0

for num in parcels:
    tif_path = os.path.join(RAW_DIR, f"{num}.tiff")
    geojson_path = os.path.join(RAW_DIR, f"geojson_{num}.geojson")

    if not (os.path.exists(tif_path) and os.path.exists(geojson_path)):
        print(f"Missing: {num}.tiff or geojson_{num}.geojson. Checking: {tif_path}")
        continue

    print(f"Processing Parcel {num}...")

    with rasterio.open(tif_path) as src:
        # UNIVERSAL CRS FIX: Ensures both files match to WGS84 if metadata is missing
        raster_crs = src.crs if src.crs is not None else "EPSG:4326"

        gdf = gpd.read_file(geojson_path)
        if gdf.crs is None:
            gdf.set_crs(epsg=4326, inplace=True)
        gdf = gdf.to_crs(raster_crs)

        valid_geoms = [geom for geom in gdf.geometry if geom is not None and geom.is_valid]

        if not valid_geoms:
            print(f"No valid geometries found for {num}.")
            continue

        geom_pixels = [(geom, 1) for geom in valid_geoms]   # Creating full binary mask
        full_mask = rasterize(
            geom_pixels,
            out_shape=(src.height, src.width),
            transform=src.transform,
            fill=0,
            dtype=np.uint8
        )

        #SLICING
        for y in range(0, src.height - patch_h, patch_h):
            for x in range(0, src.width - patch_w, patch_w):
                window = Window(x, y, patch_w, patch_h)

                img_patch = np.moveaxis(src.read([1, 2, 3], window=window), 0, -1)
                mask_patch = full_mask[y:y+patch_h, x:x+patch_w]

                # Only save if Grass in the patch
                if np.sum(mask_patch) == 0:
                    continue

                file_name = f"parcel_{num}_tile_{x}_{y}.png"

                cv2.imwrite(os.path.join(PATCH_IMG_DIR, file_name), cv2.cvtColor(img_patch, cv2.COLOR_RGB2BGR))
                cv2.imwrite(os.path.join(PATCH_MASK_DIR, file_name), mask_patch * 255)

                total_patches_saved += 1

print(f"\nSuccess! Generated {total_patches_saved} training pairs.")

In [2]:
for num in parcels:
    tif_path = os.path.join(RAW_DIR, f"{num}.tiff")
    geojson_path = os.path.join(RAW_DIR, f"geojson_{num}.geojson")

    if not (os.path.exists(tif_path) and os.path.exists(geojson_path)):
        continue

    print(f"🔄 Processing Parcel {num} in Pixel Space...")

    with rasterio.open(tif_path) as src:
        gdf = gpd.read_file(geojson_path)

        valid_geoms = [geom for geom in gdf.geometry if geom.is_valid]

        # Use a default identity transform (pixel-by-pixel)
        full_mask = rasterize(
            [(geom, 1) for geom in valid_geoms],
            out_shape=(src.height, src.width),
            transform=rasterio.Affine.identity(),
            fill=0,
            dtype=np.uint8
        )

In [2]:
import geopandas as gpd

geojson_path = "/content/drive/MyDrive/ottermap_challenge/data/raw/geojson_1.geojson"
gdf = gpd.read_file(geojson_path)
print("First 3 rows of coordinates:")
print(gdf.geometry.head(3))

# 2. Tiling & Quality ControlPatching: Large satellite images are sliced into

*   $256 \times 512$ pixel tiles to optimize GPU memory usage.
*   Smart Filtering: We discard "empty" patches (those with no grass labels) to keep the dataset focused and efficient.



In [2]:
import os
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import numpy as np
import cv2

PROJECT_ROOT = "/content/drive/MyDrive/ottermap_challenge"
RAW_DIR = os.path.join(PROJECT_ROOT, "data/raw")
PATCH_IMG_DIR = os.path.join(PROJECT_ROOT, "data/patches/images")
PATCH_MASK_DIR = os.path.join(PROJECT_ROOT, "data/patches/masks")

os.makedirs(PATCH_IMG_DIR, exist_ok=True)
os.makedirs(PATCH_MASK_DIR, exist_ok=True)

parcels = ["1", "2", "3"]
patch_h, patch_w = 256, 512
total_patches_saved = 0

for num in parcels:
    tif_path = os.path.join(RAW_DIR, f"{num}.tiff")
    geojson_path = os.path.join(RAW_DIR, f"geojson_{num}.geojson")

    if not (os.path.exists(tif_path) and os.path.exists(geojson_path)):
        continue

    print(f"🔄 Processing Parcel {num} with CRS Forced Alignment...")

    gdf = gpd.read_file(geojson_path)   # GeoJSON to Lat/Lon
    if gdf.crs is None:
        gdf.set_crs(epsg=4326, inplace=True)

    with rasterio.open(tif_path) as src:
        from rasterio.transform import from_bounds
        minx, miny, maxx, maxy = gdf.total_bounds
        transform = from_bounds(minx, miny, maxx, maxy, src.width, src.height)

        full_mask = rasterize(
            [(geom, 1) for geom in gdf.geometry],
            out_shape=(src.height, src.width),
            transform=transform,
            fill=0,
            dtype=np.uint8
        )

        # Slicing loop remains the same
        for y in range(0, src.height - patch_h, patch_h):
            for x in range(0, src.width - patch_w, patch_w):
                window = Window(x, y, patch_w, patch_h)
                img_patch = np.moveaxis(src.read([1, 2, 3], window=window), 0, -1)
                mask_patch = full_mask[y:y+patch_h, x:x+patch_w]

                if np.sum(mask_patch) > 100: # Threshold: only save if >100 pixels are grass
                    cv2.imwrite(os.path.join(PATCH_IMG_DIR, f"p{num}_{x}_{y}.png"), cv2.cvtColor(img_patch, cv2.COLOR_RGB2BGR))
                    cv2.imwrite(os.path.join(PATCH_MASK_DIR, f"p{num}_{x}_{y}.png"), mask_patch * 255)
                    total_patches_saved += 1

print(f"\nSuccess! Generated {total_patches_saved} patches.")

## Spliting into Training and testing 80% training and 20% testing
Later we will used Cross validation mechanism
First 20% testing and remain below 80% for training
second after 20% and before 40% testing remain 80 % for training
like that and Finaly we will calculate the either minium loss across all validation
or Average Of All loss

In [ ]:
import os
import shutil
import random

PROJECT_ROOT = "/content/drive/MyDrive/ottermap_challenge"
PATCH_IMG_DIR = os.path.join(PROJECT_ROOT, "data/patches/images")
PATCH_MASK_DIR = os.path.join(PROJECT_ROOT, "data/patches/masks")

os.makedirs("/content/drive/MyDrive/ottermap_challenge/data/train/images", exist_ok=True)
os.makedirs("/content/drive/MyDrive/ottermap_challenge/data/train/masks", exist_ok=True)
os.makedirs("/content/drive/MyDrive/ottermap_challenge/data/val/images", exist_ok=True)
os.makedirs("/content/drive/MyDrive/ottermap_challenge/data/val/masks", exist_ok=True)

# SPLIT DATA
files = [f for f in os.listdir(PATCH_IMG_DIR) if f.endswith('.png')]
random.shuffle(files)
split = int(0.8 * len(files))

print(f"Total images found: {len(files)}")
print(f"Splitting: {split} for training, {len(files) - split} for validation.")

for i, f in enumerate(files):
    target = "train" if i < split else "val"
    shutil.copy(os.path.join(PATCH_IMG_DIR, f), f"/content/drive/MyDrive/ottermap_challenge/data/{target}/images/{f}")
    shutil.copy(os.path.join(PATCH_MASK_DIR, f), f"/content/drive/MyDrive/ottermap_challenge/data/{target}/masks/{f}")

print("Successfully organized data into train and val folders!")

Total images found: 327
Splitting: 261 for training, 66 for validation.
🎉 Successfully organized data into train and val folders!


Creating Custom data set to connect with It uses torch.utils.data.Dataset to fetch files on the fly and converts them into tensors.

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

class GrassDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.image_dir, self.images[index])
        mask_path = os.path.join(self.mask_dir, self.images[index])

        # Load image (convert to RGB) and mask (convert to grayscale)
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Convert mask to binary [0, 1]
        mask = np.array(mask)
        mask = (mask > 127).astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0) # Add channel dim

        # Apply transforms (e.g., resizing or normalization)
        if self.transform:
            image = self.transform(image)

        return image, mask


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = GrassDataset("/content/drive/MyDrive/ottermap_challenge/data/train/images",
                        "/content/drive/MyDrive/ottermap_challenge/data/train/masks",
                        transform=transform)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True)

val_ds = GrassDataset("/content/drive/MyDrive/ottermap_challenge/data/val/images",
                      "/content/drive/MyDrive/ottermap_challenge/data/val/masks",
                      transform=transform)


val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False)

print(f"Validation loader created with {len(val_ds)} images.")

✅ Validation loader created with 118 images.


# Data Augmentation and Generalization:

*   In The Orignal Data Ther is So much Miss Labelled and Training Losses So PErfomr Data Augmentation For more Genrealize the Data To Reduce the Loss and Work on Test Dataset perfectly






In [2]:
pip install albumentations

1. HorizontalFlip & Rotate: This destroys the "half-screen" diagonal bias your model has learned. It forces the model to realize that grass is grass, no matter where it is in the picture.

2. RandomResizedCrop: This prevents the model from relying on the global composition of the image, forcing it to look at local patterns and textures instead.

3. RandomBrightnessContrast & HueSaturationValue: This creates "synthetic" lighting conditions, teaching the model that grass in shadows (dark) and grass in sunlight (bright) are the same class.

# Bringing All training data to Colab it will be 50X-100X faster

In [2]:
import shutil
import os


drive_base = "/content/drive/MyDrive/ottermap_challenge/data"
local_base = "/content/local_data"

os.makedirs(local_base, exist_ok=True)

folders_to_copy = ['train', 'val']

for folder in folders_to_copy:
    src = os.path.join(drive_base, folder)
    dst = os.path.join(local_base, folder)

    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Successfully copied: {folder}")
    else:
        print(f"CRITICAL ERROR: Source folder not found: {src}")
print("Folders now in local_data:", os.listdir("/content/local_data"))

####1.  Here Since I am Using Google Colab for Fast Running and optimize Usage of REsource by using T4-GPU and 16 gb free Ram Awailable Based on that i use this
#### 2. And Upload all My data here THat reduce the Bottleneck speed of Training ,instead of accessing each batch of data from drive it was taking Too Much time so I did this and My laptop was having Only 8gb of Ram
####3.  In Data Augementation process I am Trying to make data more Generalize by Horizontal Flipping, Rotating,Texture ,Color,Saturation in such way it can Detect all type of Pattern in few Dataset
####Like Detecting Breed Of Horse if very Difficult but Data Augmentation helo Alot

In [2]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset,DataLoader
import cv2

class GrassDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.image_dir, self.images[index])
        mask_path = os.path.join(self.mask_dir, self.images[index])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        # mask is already a tensor here, so we use .float() and create a boolean mask then convert to float
        mask = (mask > 127).float()
        mask = mask.unsqueeze(0)

        return image, mask

transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.5),
    A.RandomResizedCrop(height=256, width=256, size=(256, 256), scale=(0.8, 1.0), p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_img_dir = "/content/local_data/train/images"
train_mask_dir = "/content/local_data/train/masks"
train_ds = GrassDataset(train_img_dir, train_mask_dir, transform=transform)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)


val_img_dir = "/content/local_data/val/images"
val_mask_dir = "/content/local_data/val/masks"
val_ds = GrassDataset(val_img_dir, val_mask_dir, transform=transform)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

## If You  want to Access All training data and Validation data fromDrive then Just Replace colab Dir to This dir structure that given Below

In [2]:
img_dir = "/content/drive/MyDrive/ottermap_challenge/data/train/images"
mask_dir = "/content/drive/MyDrive/ottermap_challenge/data/train/masks"
train_ds = GrassDataset(img_dir, mask_dir, transform=transform)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)


val_img_dir="/content/drive/MyDrive/ottermap_challenge/data/val/images"
val_mask_dir="/content/drive/MyDrive/ottermap_challenge/data/val/masks"

#No augmentation, only scaling/normalization
val_transform = A.Compose([
    A.Resize(256, 256), # Resize to match training input
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_ds = GrassDataset(val_img_dir, val_mask_dir, transform=val_transform)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)


In [2]:
!pip install segmentation-models-pytorch

Resnet 34 Pretrained model

In [2]:
# Test just one batch
try:
    data_iter = iter(train_loader)
    images, masks = next(data_iter)
    print("Success! First batch loaded. Shape:", images.shape)
except Exception as e:
    print("Error loading data:", e)

#4. Model Training
##UNet Model For Pixel level Classification


Here i used Unet model
iterate through your data, calculate performance, and automatically save the best model to your Google Drive


### 1. Downsampling (Feature Extraction)
Downsampling reduces the width and height of the feature maps, effectively increasing the "receptive field" of the model. This allows the network to capture high-level contextual information.
"Stride" ,"Filter", "Padding","Activation Function", "Conv Pooling layer"
"Batch Norm"
### 2. Upsampling (Feature Reconstruction)
Upsampling restores the spatial resolution of the feature maps to the original input size, allowing the model to make pixel-level predictions (the "where").
There are Severak Pooling Type


*   Max Pooling only max value extract from feature img
*   Min Pooling Only min value extrac from Featue imag
*  Avg Pooling
 *Formula for Finding Feature img or Extracting Pooing:
  ((K+2P-N)/S)+1


---


1.   U-Net for Perform pixel-level classification
2.   ResNet Extract high-level visual features
3.   ImageNet Leverage pre-learned edge/texture detection
## Resnet for Extract high-level visual features

*   ResNet (Residual Network) is a deep convolutional architecture that uses "shortcut" connections to solve the vanishing gradient problem in deep networks
*   List item

## IMageNet Leverage pre-learned edge/texture detection
1. Instead of starting with random, "ignorant" weights, your ResNet34 starts with weights pre-trained on ImageNet (a massive database of over 14 million images).

## Dice Loss:
Handle class imbalance/maximize overlap








In [21]:
import torch
import segmentation_models_pytorch as smp
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(device)

loss_fn = smp.losses.DiceLoss(mode='binary')
optimizer = optim.Adam(model.parameters(), lr=1e-4)


best_loss = float('inf')
save_path = "/content/drive/MyDrive/ottermap_challenge/models/model_v2.pth"

print("Starting training...")

for epoch in range(50):
    model.train()
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            val_loss += loss_fn(outputs, masks).item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1}, Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), save_path)
        print("Model saved to Drive!")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Starting training...
Epoch 1, Val Loss: 0.3226
Model saved to Drive!
Epoch 2, Val Loss: 0.2788
Model saved to Drive!
Epoch 3, Val Loss: 0.2731
Model saved to Drive!
Epoch 4, Val Loss: 0.2447
Model saved to Drive!
Epoch 5, Val Loss: 0.2469
Epoch 6, Val Loss: 0.2263
Model saved to Drive!
Epoch 7, Val Loss: 0.2064
Model saved to Drive!
Epoch 8, Val Loss: 0.2163
Epoch 9, Val Loss: 0.2803
Epoch 10, Val Loss: 0.2300
Epoch 11, Val Loss: 0.2068
Epoch 12, Val Loss: 0.1925
Model saved to Drive!
Epoch 13, Val Loss: 0.1767
Model saved to Drive!
Epoch 14, Val Loss: 0.2022
Epoch 15, Val Loss: 0.2122
Epoch 16, Val Loss: 0.1820
Epoch 17, Val Loss: 0.1945
Epoch 18, Val Loss: 0.1773
Epoch 19, Val Loss: 0.1669
Model saved to Drive!
Epoch 20, Val Loss: 0.1889
Epoch 21, Val Loss: 0.1923
Epoch 22, Val Loss: 0.1671
Epoch 23, Val Loss: 0.1702
Epoch 24, Val Loss: 0.1689
Epoch 25, Val Loss: 0.1696
Epoch 26, Val Loss: 0.1713
Epoch 27, Val Loss: 0.1749
Epoch 28, Val Loss: 0.1719
Epoch 29, Val Loss: 0.1608
Model s

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

# SAving Different Model in Different Scenario
model Weight of Different Version mean at       

####**** CHANGE VERSION MANUALLY*****
##### 1. Fine Tunning with YOLOv8-seg,U-Net,SegFormer,SAM
#####2. Before Data Augmentation and After Data Augementation weight
#####3. After Hyper parameter Tunning



In [2]:
import os
import torch
model_base_dir = "/content/drive/MyDrive/ottermap_challenge/models"
os.makedirs(model_base_dir, exist_ok=True)

version = "v2"  # Change this to v1, v2, v3, etc. for Each model Save
save_path = os.path.join(model_base_dir, f"model_{version}.pth")
torch.save(model.state_dict(), save_path)
print(f"Model saved successfully to: {save_path}")

Load The Trained Model Weight into Google Colab

In [3]:
import torch
import segmentation_models_pytorch as smp

model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1)
checkpoint_path = "/content/drive/MyDrive/ottermap_challenge/best_model.pth"

model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
model.eval()
print("Model loaded successfully!")

Validating IMage Whetehr Grass land, Road, Street, Playground,Scool, Residential, Tree what object are there

In [3]:
import torch
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None, # loading Our trained weights ONly
    in_channels=3,
    classes=1
)

checkpoint = torch.load("/content/drive/MyDrive/ottermap_challenge/models/model_v2.pth", map_location='cpu')
model.load_state_dict(checkpoint, strict=True)  # weights
model.eval()
print("Weights loaded successfully!")

#We can test Our Image Directly from Drive or bring to colab then testing
### As here we Only Classifiy Grass or not Grass so i used " SIGMOID " Function TO triggered White for Grass and Black for Non Grass  Land

In [3]:

import matplotlib.pyplot as plt
model.eval()
with torch.no_grad():
    images, masks = next(iter(val_loader))
    images = images.to(device)

    mask_pred = model(images[3].unsqueeze(0))
    test_img = images[0].cpu().permute(1, 2, 0).numpy()
    mask_pred = torch.sigmoid(mask_pred) > 0.5


plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(test_img); plt.title("Original Image")
plt.subplot(1, 2, 2); plt.imshow(mask_pred.squeeze().cpu().numpy(), cmap='gray'); plt.title("Predicted Mask")
plt.show()
print(mask_pred)
prediction=mask_pred

In [3]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# test_img = Image.open("/content/drive/MyDrive/ottermap_challenge/data/val/images/p3_5120_5120.png").convert("RGB")  # for
test_img = Image.open("/content/local_data/val/images/p3_5120_5120.png").convert("RGB")
input_tensor = transform(test_img).unsqueeze(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
input_tensor = input_tensor.to(device)

print(f"Input Min: {input_tensor.min():.4f}, Max: {input_tensor.max():.4f}")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(test_img); plt.title("Original Image")
plt.subplot(1, 2, 2); plt.imshow(mask_pred.squeeze().numpy(), cmap='gray'); plt.title("Predicted Mask")
plt.show()

In [40]:
print(f"Model is on: {next(model.parameters()).device}")
print(f"Input is on: {input_tensor.device}")

Model is on: cuda:0
Input is on: cuda:0


### Visualize the 3 Training sample  and their mask properly alignd or not
And we Get some our image not Properly Label That make model so Confusion and
getting stuck while in training and Produce higher Loss as grass is represented black nean Loss is very high for that image andif we have many more like that of image that will be Ambiguity so
# I put all Not labeled Properly image and mask in Quorantine Folder

In [3]:

import matplotlib.pyplot as plt
import os

img_dir = "/content/drive/MyDrive/ottermap_challenge/data/train/images/"
mask_dir = "/content/drive/MyDrive/ottermap_challenge/data/train/masks/"

sample_files = os.listdir(img_dir)[:3]

fig, axes = plt.subplots(3, 2, figsize=(10, 10))
for i, file in enumerate(sample_files):
    img = plt.imread(os.path.join(img_dir, file))
    mask = plt.imread(os.path.join(mask_dir, file)) # ensure file names match
    axes[i, 0].imshow(img)
    axes[i, 1].imshow(mask, cmap='gray')
plt.show()

### Model is Sufferring from Spatial Bias(Overfitting) and Data Label Mismatch

#### 1.Spatial Bias:-
samples were likely not centered, grass was mostly in one corner (bottom-left). The model has "learned" that grass only exists in that specific region of the image frame, rather than learning the texture of grass itself.
Blindly Apply learned mask
#### 1. Label Mismatch:-
model predicts black for grass,learned that grass = background.
training masks were labeled inconsistently—for
eg:some grass patches were labeled as "1" (white) but others were accidentally left as "0" (black/background).
#### 3. Removing Wrongly Labeled Dataset that confuse model (Garbage Out Tarika)
   a) Eliminating Ambiguity: Two image with same texture but Different Label
   b) Preventing Model lazyness: model ignore that confusion just memorize
### Solution Finding where data have highest Loss That will be wrongly labeled data and PLace it To "Quarantine Folder"
Only keeping if Loss is high but Correctly Classified LAbel data
and all Correctly labeld data with min loss


# Here i am showing the image with Higher Loss mean they are not Labeled Perfectly

In [3]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
criterion = torch.nn.BCEWithLogitsLoss()

audit_loader = DataLoader(train_ds, batch_size=1, shuffle=False)
loss_tracking = []

print("Running loss audit...")
with torch.no_grad():
    for i, (images, masks) in enumerate(audit_loader):
        images, masks = images.to(device), masks.to(device)
        output = model(images)
        loss = criterion(output, masks)

        filename = train_ds.images[i]
        loss_tracking.append((filename, loss.item()))

loss_tracking.sort(key=lambda x: x[1], reverse=True)    # Sorting by loss descending

# Print the top 10 worst offenders
print("\nTop 10 samples with the highest loss:")
for i in range(min(10, len(loss_tracking))):
    fname, loss_val = loss_tracking[i]
    print(f"File: {fname}, Loss: {loss_val:.4f}")

## Mask Matching to image or not ?

In [3]:
import matplotlib.pyplot as plt
import os

# Define paths
image_folder = "/content/drive/MyDrive/ottermap_challenge/data/train/images/"
mask_folder = "/content/drive/MyDrive/ottermap_challenge/data/train/masks/"

# Get a list of image filenames
files = sorted(os.listdir(image_folder))

# Show the first 5 pairs
for i in range(min(5, len(files))):
    img_name = files[i]
    mask_name = img_name  # Ensure this matches your naming convention

    img = plt.imread(os.path.join(image_folder, img_name))
    mask = plt.imread(os.path.join(mask_folder, mask_name))

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(img)
    ax[0].set_title(f"Image: {img_name}")
    ax[1].imshow(mask, cmap='gray')
    ax[1].set_title(f"Mask: {mask_name}")
    plt.show()

In [3]:
import matplotlib.pyplot as plt

# Get the raw probabilities
probs = torch.sigmoid(prediction).squeeze().cpu().numpy()

# Plot the heatmap
plt.figure(figsize=(6, 6))
plt.imshow(probs, cmap='jet', vmin=0, vmax=1)
plt.colorbar(label='Probability of Grass')
plt.title("Grass Probability Heatmap")
plt.show()

##Keep Miss Lablled Data into Quarantine Train->Quarantine Folder

In [3]:
import os
import shutil
import torch
from torch.utils.data import DataLoader

img_dir = "/content/drive/MyDrive/ottermap_challenge/data/train/images"
mask_dir = "/content/drive/MyDrive/ottermap_challenge/data/train/masks"
quarantine_dir = "/content/drive/MyDrive/ottermap_challenge/quarantine"

os.makedirs(quarantine_dir, exist_ok=True)
audit_loader = DataLoader(train_ds, batch_size=1, shuffle=False)
model.eval()
criterion = torch.nn.BCEWithLogitsLoss()

loss_tracking = []

print("Analyzing training data for high-loss samples...")
with torch.no_grad():
    for i, (images, masks) in enumerate(audit_loader):
        images, masks = images.to(device), masks.to(device)
        output = model(images)
        loss = criterion(output, masks)
        filename = train_ds.images[i]
        loss_tracking.append((filename, loss.item()))

loss_tracking.sort(key=lambda x: x[1], reverse=True)

# Quarantine Top 50 Offenders
num_to_quarantine = 50
print(f"Moving top {num_to_quarantine} worst samples to: {quarantine_dir}")

for i in range(min(num_to_quarantine, len(loss_tracking))):
    fname, lval = loss_tracking[i]

    src_img = os.path.join(img_dir, fname)
    src_mask = os.path.join(mask_dir, fname)
    dst_img = os.path.join(quarantine_dir, fname)
    dst_mask = os.path.join(quarantine_dir, "mask_" + fname)
    if os.path.exists(src_img):
        shutil.move(src_img, dst_img)

    if os.path.exists(src_mask):
        shutil.move(src_mask, dst_mask)

    print(f"Quarantined: {fname} (Loss: {lval:.4f})")

print("\nAudit Complete. Your training folder is now cleaned of these 50 samples.")